# Environment Setup — SegTran (Optic Cup)

This notebook creates and configures the conda `segtran` environment needed to run the following notebooks:
- `05_preprocessing_OpticCup_REFUGE.ipynb`
- `06_train_inference_SegTran_OpticCup.ipynb`

**Run this notebook once** with the default kernel (`Python 3 (ipykernel)`) before using the others.

---

### What this notebook does

| Step | Description |
|:-----:|:----------|
| 1 | Recreates the conda `segtran` environment with Python 3.10 |
| 2 | Installs PyTorch/SegTran dependencies (`requirements.txt`) |
| 3 | Installs additional packages (TensorFlow, OpenCV, scikit-image) for MNet_DeepCDR |
| 4 | Registers the Jupyter `segtran` kernel |
| 5 | Verifies imports and repository paths |

**After completing**, select the `segtran` kernel in the following notebooks.

In [4]:
import os
import subprocess
import sys
from pathlib import Path

# ─── Paths ─────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = Path('/home/bruno-borges/soft_dice_confidence/tasks_models/optic_cup')
SEGTRAN_DIR   = NOTEBOOK_DIR / 'segtran'
CODE_DIR      = SEGTRAN_DIR / 'code'
REQUIREMENTS  = NOTEBOOK_DIR / 'requirements.txt'
CONDA_BIN     = Path('/home/bruno-borges/miniconda3/bin/conda')
ENV_NAME      = 'segtran'
PYTHON_VER    = '3.10'

# Check essential paths
checks = {
    'segtran directory'    : SEGTRAN_DIR,
    'code directory'       : CODE_DIR,
    'requirements.txt'     : REQUIREMENTS,
    'conda'                : CONDA_BIN,
}

all_ok = True
for label, path in checks.items():
    status = '✓' if path.exists() else '✗ NOT FOUND'
    all_ok = all_ok and path.exists()
    print(f'  [{status}] {label}: {path}')

if not all_ok:
    raise RuntimeError('Essential paths missing. Check the repository structure.')

print('\n✓ Paths verified.')

  [✓] segtran directory: /home/bruno-borges/soft_dice_confidence/tasks_models/optic_cup/segtran
  [✓] code directory: /home/bruno-borges/soft_dice_confidence/tasks_models/optic_cup/segtran/code
  [✓] requirements.txt: /home/bruno-borges/soft_dice_confidence/tasks_models/optic_cup/requirements.txt
  [✓] conda: /home/bruno-borges/miniconda3/bin/conda

✓ Paths verified.


## 1. Creating the conda `segtran` environment

Removes the previous environment (if it exists and is incomplete) and recreates it with Python 3.10.

> **Note**: Python 3.10 is required — `segmentation-models-pytorch` requires `>=3.10`.

In [5]:
%%bash
set -e

CONDA=/home/bruno-borges/miniconda3/bin/conda
ENV_NAME=segtran
PYTHON_VER=3.10

PYTHON_BIN=/home/bruno-borges/miniconda3/envs/${ENV_NAME}/bin/python

# Check if Python exists AND is >= 3.10
NEEDS_RECREATE=true
if [ -x "$PYTHON_BIN" ]; then
    MAJOR=$($PYTHON_BIN -c "import sys; print(sys.version_info.major)")
    MINOR=$($PYTHON_BIN -c "import sys; print(sys.version_info.minor)")
    if [ "$MAJOR" -ge 3 ] && [ "$MINOR" -ge 10 ]; then
        echo "[INFO] Environment ${ENV_NAME} already uses Python $($PYTHON_BIN --version) — nothing to do."
        NEEDS_RECREATE=false
    else
        echo "[INFO] Environment ${ENV_NAME} uses Python $($PYTHON_BIN --version) — needs to be recreated with Python ${PYTHON_VER}."
    fi
fi

if [ "$NEEDS_RECREATE" = true ]; then
    echo "[INFO] Removing existing environment '${ENV_NAME}'..."
    $CONDA env remove -n $ENV_NAME -y 2>/dev/null || true
    echo "[INFO] Creating environment '${ENV_NAME}' com Python ${PYTHON_VER}..."
    $CONDA create -n $ENV_NAME python=$PYTHON_VER -y
    echo "✓ Ambiente ${ENV_NAME} created with $(/home/bruno-borges/miniconda3/envs/${ENV_NAME}/bin/python --version)."
fi

[INFO] Environment segtran already uses Python Python 3.10.20 — nothing to do.


## 2. Installing SegTran dependencies

Installs packages listed in `requirements.txt` (PyTorch, timm, imgaug, etc.).

> **numpy**: pinned at `1.26.4` — compatibility version between TF 2.17 (`<2.0`) and torch.

In [6]:
%%bash
set -e

PIP=/home/bruno-borges/miniconda3/envs/segtran/bin/pip

echo "[INFO] Installing PyTorch (CUDA 11.8)..."
$PIP install torch torchvision --index-url https://download.pytorch.org/whl/cu118 --quiet
echo "✓ PyTorch + torchvision installed."

echo "[INFO] Installing SegTran dependencies..."
$PIP install \
    tqdm \
    tensorboardX \
    thop \
    'timm>=0.4' \
    imgaug \
    ml_collections \
    easydict \
    nibabel \
    medpy \
    h5py \
    fvcore \
    --quiet

$PIP install git+https://github.com/qubvel/segmentation_models.pytorch --quiet

# Pin numpy to 1.26.4: compatible with TF 2.17 (<2.0) and torch
$PIP install 'numpy==1.26.4' --quiet

echo "✓ SegTran dependencies installed."

[INFO] Installing PyTorch (CUDA 11.8)...
✓ PyTorch + torchvision installed.
[INFO] Installing SegTran dependencies...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.1 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.2.6 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


✓ SegTran dependencies installed.


## 3. Installing additional packages (MNet_DeepCDR)

The preprocessing `05_preprocessing_OpticCup_REFUGE.ipynb` uses TensorFlow (via MNet_DeepCDR),  
OpenCV e scikit-image.

> **opencv**: pinned at `4.8.1.78` — last version that accepts numpy 1.x.  
> **tensorflow**: `2.17` — works with numpy 1.26.4 (numpy 2.x causa `SystemError` no TF).

In [7]:
%%bash
set -e

PIP=/home/bruno-borges/miniconda3/envs/segtran/bin/pip

echo "[INFO] Installing TensorFlow 2.17..."
$PIP install 'tensorflow==2.17.*' --quiet
echo "✓ TensorFlow installed."

echo "[INFO] Installing OpenCV 4.8 (compatible with numpy 1.x)..."
$PIP install 'opencv-python-headless==4.8.1.78' --quiet
echo "✓ OpenCV installed."

echo "[INFO] Installing scikit-image, scikit-learn, matplotlib, pandas, Pillow..."
$PIP install scikit-image scikit-learn matplotlib pandas Pillow --quiet

# Ensure numpy 1.26.4 after TF installation (which may attempt a downgrade)
$PIP install 'numpy==1.26.4' --quiet

echo "✓ Additional packages installed."

[INFO] Installing TensorFlow 2.17...
✓ TensorFlow installed.
[INFO] Installing OpenCV 4.8 (compatible with numpy 1.x)...
✓ OpenCV installed.
[INFO] Installing scikit-image, scikit-learn, matplotlib, pandas, Pillow...
✓ Additional packages installed.


## 4. Registering the Jupyter `segtran` kernel

Registers the environment as a selectable kernel in Jupyter Lab / Notebook.

In [8]:
%%bash
set -e

PYTHON=/home/bruno-borges/miniconda3/envs/segtran/bin/python

# Ensure ipykernel is installed in the env
$PYTHON -m pip install ipykernel --quiet

# Register the kernel
$PYTHON -m ipykernel install --user --name=segtran --display-name='Python (segtran)'

echo ""
echo "✓ Kernel 'segtran' registered."
echo "  Select 'Python (segtran)' in notebooks 05 and 06."

Installed kernelspec segtran in /home/bruno-borges/.local/share/jupyter/kernels/segtran

✓ Kernel 'segtran' registered.
  Select 'Python (segtran)' in notebooks 05 and 06.


## 5. Import verification

In [9]:
%%bash

PYTHON=/home/bruno-borges/miniconda3/envs/segtran/bin/python

echo "=== Import verification in segtran env ==="
echo ""

packages=(
    "torch"
    "torchvision"
    "timm"
    "imgaug"
    "ml_collections"
    "easydict"
    "nibabel"
    "medpy"
    "h5py"
    "fvcore"
    "tensorboardX"
    "thop"
    "tensorflow"
    "cv2"
    "skimage"
    "PIL"
    "matplotlib"
    "sklearn"
    "segmentation_models_pytorch"
)

all_ok=true
for pkg in "${packages[@]}"; do
    if $PYTHON -c "import $pkg" 2>/dev/null; then
        version=$($PYTHON -c "import $pkg; print(getattr($pkg, '__version__', 'ok'))" 2>/dev/null)
        echo "  [✓] $pkg  $version"
    else
        echo "  [✗] $pkg  — NOT FOUND"
        all_ok=false
    fi
done

echo ""
if $all_ok; then
    echo "✓ All imports verified."
else
    echo "⚠ Some packages were not found. Check the previous cells."
fi

=== Import verification in segtran env ===

  [✓] torch  2.7.1+cu118
  [✓] torchvision  0.22.1+cu118
  [✓] timm  1.0.27
  [✓] imgaug  0.4.0
  [✓] ml_collections  1.1.0
  [✓] easydict  ok
  [✓] nibabel  5.4.2
  [✓] medpy  0.5.2
  [✓] h5py  3.16.0
  [✓] fvcore  0.1.5.post20221221
  [✓] tensorboardX  2.6.5
  [✓] thop  0.1.1
  [✓] tensorflow  2.17.1
  [✓] cv2  4.8.1
  [✓] skimage  0.25.2
  [✓] PIL  12.2.0
  [✓] matplotlib  3.10.9
  [✓] sklearn  1.7.2
  [✓] segmentation_models_pytorch  0.5.1.dev0

✓ All imports verified.


In [10]:
%%bash

PYTHON=/home/bruno-borges/miniconda3/envs/segtran/bin/python

echo "=== PyTorch — GPU available? ==="
$PYTHON - <<'EOF'
import torch
print(f'  torch version : {torch.__version__}')
print(f'  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
    print(f'  CUDA version   : {torch.version.cuda}')
EOF

=== PyTorch — GPU available? ===
  torch version : 2.7.1+cu118
  CUDA available: True
  GPU            : NVIDIA GeForce RTX 3090
  CUDA version   : 11.8


## 6. SegTran repository path verification

In [11]:
import os
from pathlib import Path

NOTEBOOK_DIR = Path('/home/bruno-borges/soft_dice_confidence/tasks_models/optic_cup')
SEGTRAN_DIR  = NOTEBOOK_DIR / 'segtran'
CODE_DIR     = SEGTRAN_DIR / 'code'
DATA_DIR     = SEGTRAN_DIR / 'data' / 'fundus'
MNET_DIR     = CODE_DIR / 'MNet_DeepCDR'
DEEP_MODEL   = MNET_DIR / 'deep_model'

print('=== SegTran repository structure ===\n')

dirs_to_check = {
    'segtran/'                          : SEGTRAN_DIR,
    'segtran/code/'                     : CODE_DIR,
    'segtran/code/MNet_DeepCDR/'        : MNET_DIR,
    'segtran/code/MNet_DeepCDR/deep_model/' : DEEP_MODEL,
    'segtran/data/fundus/'              : DATA_DIR,
}

for label, path in dirs_to_check.items():
    status = '✓' if path.exists() else '✗ MISSING'
    print(f'  [{status}] {label}')

print()

# Scripts principais
scripts = ['train2d.py', 'test2d.py', 'train2d.sh', 'test3d.py']
print('Scripts SegTran:')
for s in scripts:
    p = CODE_DIR / s
    status = '✓' if p.exists() else '—'
    print(f'  [{status}] {s}')

print()

# Pre-trained models MNet
print('MNet_DeepCDR pre-trained models:')
for model in ['Model_DiscSeg_ORIGA.h5', 'Model_MNet_REFUGE.h5']:
    p = DEEP_MODEL / model
    status = '✓' if p.exists() else '✗ MISSING — download from https://github.com/HzFu/MNet_DeepCDR'
    print(f'  [{status}] {model}')

print()

# Preprocessed fundus data
print('Preprocessed datasets (fundus):')
for ds in ['refuge_train', 'refuge_test', 'origa', 'g1020']:
    img_dir = DATA_DIR / ds / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*.png')))
        status = f'{n} images' if n > 0 else '✗ empty'
    else:
        status = '✗ missing — run 05_preprocessing'
    print(f'  {ds:20s}: {status}')

=== SegTran repository structure ===

  [✓] segtran/
  [✓] segtran/code/
  [✓] segtran/code/MNet_DeepCDR/
  [✓] segtran/code/MNet_DeepCDR/deep_model/
  [✓] segtran/data/fundus/

Scripts SegTran:
  [✓] train2d.py
  [✓] test2d.py
  [✓] train2d.sh
  [✓] test3d.py

MNet_DeepCDR pre-trained models:
  [✗ MISSING — download from https://github.com/HzFu/MNet_DeepCDR] Model_DiscSeg_ORIGA.h5
  [✗ MISSING — download from https://github.com/HzFu/MNet_DeepCDR] Model_MNet_REFUGE.h5

Preprocessed datasets (fundus):
  refuge_train        : ✗ missing — run 05_preprocessing
  refuge_test         : ✗ missing — run 05_preprocessing
  origa               : ✗ missing — run 05_preprocessing
  g1020               : ✗ missing — run 05_preprocessing


## 7. Available Jupyter kernels

In [12]:
%%bash
/home/bruno-borges/miniconda3/bin/jupyter kernelspec list

Available kernels:
  segtran         /home/bruno-borges/.local/share/jupyter/kernels/segtran
  shifts_mswml    /home/bruno-borges/.local/share/jupyter/kernels/shifts_mswml
  python3         /home/bruno-borges/miniconda3/share/jupyter/kernels/python3


## Summary

```
Conda environment: segtran  (Python 3.10)
Jupyter kernel     : Python (segtran)
Repository         : .../optic_cup/segtran/

Key packages       :
  torch              2.7.1+cu118
  tensorflow         2.17.1
  numpy              1.26.4        ← TF+torch compatibility pin
  opencv-python-headless  4.8.1.78 ← last version with support for numpy 1.x
  segmentation-models-pytorch  0.5.1.dev0
```

### Next steps

1. **Place the pre-trained models** em `segtran/code/MNet_DeepCDR/deep_model/`:
   - `Model_DiscSeg_ORIGA.h5`
   - `Model_MNet_REFUGE.h5`
   - Fonte: https://github.com/HzFu/MNet_DeepCDR

2. **Select the `Python (segtran)` kernel** in the following notebooks.

3. **Run** `05_preprocessing_OpticCup_REFUGE.ipynb` → then `06_train_inference_SegTran_OpticCup.ipynb`.